# 📘 Day 1: 데이터 수집과 텍스트 전처리

## 학습 목표
- 웹 스크래핑으로 한국어 텍스트 데이터를 직접 수집할 수 있다
- 한국어 텍스트의 특성을 이해하고, 형태소 분석 기반 전처리 파이프라인을 구축할 수 있다
- TF-IDF로 텍스트를 벡터화하고, 키워드를 추출하여 워드클라우드로 시각화할 수 있다

## 오늘의 흐름
| 내용 |
|------|
| 환경 세팅 + NLP 오리엔테이션 |
| 웹 스크래핑으로 뉴스 데이터 수집 |
| 한국어 전처리 파이프라인 |
| TF-IDF + 워드클라우드 + Streamlit 앱 |

---


## 1. 환경 세팅

### 1.1 필수 패키지 설치

Colab에서는 `pip install`로 바로 설치할 수 있습니다.  
설치에 1~2분 정도 걸립니다.


In [ ]:
# 필수 패키지 설치
# kiwipiepy: 한국어 형태소 분석기 (pip 한 줄로 설치, OS 무관)
# wordcloud: 워드클라우드 시각화

!pip install kiwipiepy wordcloud feedparser -q

print("✅ 설치 완료!")


### 1.2 패키지 임포트

오늘 사용할 주요 도구들을 미리 불러옵니다.


In [ ]:
# 데이터 수집
import requests                          # 웹 페이지 요청
from bs4 import BeautifulSoup           # HTML 파싱

# 데이터 처리
import pandas as pd                      # 데이터프레임
import re                                # 정규표현식

# 한국어 NLP
from kiwipiepy import Kiwi              # 한국어 형태소 분석기

# 시각화
import matplotlib.pyplot as plt          # 그래프
from wordcloud import WordCloud          # 워드클라우드
from collections import Counter          # 빈도 카운트

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# 한글 폰트 설정 (Colab용)
import matplotlib.font_manager as fm
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# Kiwi 초기화
kiwi = Kiwi()

print("✅ 모든 패키지 로딩 완료!")


### 1.3 NLP란?

**자연어처리(Natural Language Processing, NLP)** 는 컴퓨터가 인간의 언어를 이해하고 처리하는 기술입니다.

우리가 이전에 배운 **Computer Vision**과 비교하면:

| | Computer Vision | NLP |
|---|---|---|
| **입력** | 이미지 (픽셀 → 이미 숫자) | 텍스트 (문자 → 숫자로 변환 필요) |
| **전처리** | 리사이즈, 정규화 | 토큰화, 형태소 분석, 불용어 제거 |
| **핵심 과제** | 이미지 분류, 객체 탐지 | 텍스트 분류, 감성 분석, 검색 |

> 💡 CV에서는 이미지가 이미 숫자(픽셀값)였지만, NLP에서는 **텍스트를 숫자로 바꾸는 것** 자체가 핵심 과제입니다.

---


## 2. 웹 스크래핑으로 뉴스 데이터 수집
### 2.1 웹 스크래핑이란?

웹 스크래핑은 웹 페이지에서 원하는 데이터를 자동으로 추출하는 기술입니다.

**작동 원리:**
1. `requests`로 웹 페이지를 요청한다
2. `BeautifulSoup`으로 HTML을 파싱한다
3. 원하는 태그에서 텍스트를 추출한다

### 2.2 웹 페이지 요청해보기

먼저 간단한 웹 페이지를 요청해서 HTML 구조를 확인해봅시다.


In [ ]:
# 위키피디아 '자연어 처리' 페이지를 요청합니다
url = "https://ko.wikipedia.org/wiki/자연어_처리"

# User-Agent 헤더를 추가해야 정상적으로 응답을 받을 수 있습니다
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)

# 응답 코드 확인 (200이면 성공)
print(f"응답 코드: {response.status_code}")

# BeautifulSoup으로 HTML 파싱
soup = BeautifulSoup(response.text, 'html.parser')

# 페이지 제목 확인
if soup.title:
    print(f"페이지 제목: {soup.title.text}")
else:
    print("페이지 제목을 찾을 수 없습니다.")

### 2.3 HTML에서 원하는 정보 추출하기

웹 페이지는 HTML 태그로 구성되어 있습니다.  
`<p>` 태그는 단락, `<h2>` 태그는 소제목을 나타냅니다.


In [ ]:
# 위키피디아 본문에서 모든 단락(<p> 태그)을 추출합니다
paragraphs = soup.find_all('p')

# 처음 3개 단락만 확인
for i, para in enumerate(paragraphs[:3]):
    text = para.get_text().strip()  # 태그를 제거하고 텍스트만 추출
    if text:  # 빈 문자열이 아닌 경우만 출력
        print(f"[단락 {i+1}]")
        print(text[:200])  # 앞 200자만 출력
        print()


### 2.4 네이버 뉴스 수집 함수 만들기

이제 실제로 네이버 뉴스 검색 결과에서 기사를 수집하는 함수를 만들어봅시다.

> ⚠️ 웹 스크래핑 시 주의: 웹사이트의 이용약관과 robots.txt를 확인하고, 과도한 요청은 피해야 합니다. 학습 목적으로 소량만 수집합니다.


In [ ]:
# ============================================
# 네이버 뉴스 RSS 피드 활용 (안정적)
# ============================================
import feedparser

def search_news_rss(query, num_items=30):
    """
    네이버 뉴스 RSS 피드에서 기사를 수집하는 함수 (HTML 구조 변경에 영향 없음)

    Parameters:
        query (str): 검색어
        num_items (int): 수집할 기사 수

    Returns:
        pd.DataFrame: 제목, 요약, 링크가 담긴 데이터프레임
    """
    import urllib.parse
    encoded_query = urllib.parse.quote(query)
    url = f"https://news.google.com/rss/search?q={encoded_query}+when:7d&hl=ko&gl=KR&ceid=KR:ko"

    feed = feedparser.parse(url)

    articles = []
    for entry in feed.entries[:num_items]:
        articles.append({
            'title': entry.get('title', ''),
            'description': entry.get('summary', ''),
            'link': entry.get('link', ''),
            'query': query,
        })

    return pd.DataFrame(articles)

### 2.5 여러 주제의 뉴스 수집하기

각자 관심 있는 주제를 골라서 뉴스를 수집해봅시다.  
수집한 데이터는 이후 수업에서 계속 활용합니다.


In [ ]:
# 여러 주제로 뉴스를 수집합니다
# 💡 원하는 검색어로 자유롭게 변경하세요!
search_queries = ["인공지능", "기후변화", "반도체", "K팝", "부동산"]

all_news = []
for query in search_queries:
    print(f"🔍 '{query}' 검색 중...")
    df_temp = search_news_rss(query)  # 3페이지씩 (약 30개 기사)
    all_news.append(df_temp)
    print(f"   → {len(df_temp)}개 기사 수집")

# 전체 데이터프레임으로 합치기
df_news = pd.concat(all_news, ignore_index=True)
print(f"\n✅ 총 {len(df_news)}개 기사 수집 완료!")
print(f"주제별 분포:\n{df_news['query'].value_counts()}")


In [ ]:
# 수집한 데이터 확인
df_news.head(10)


In [ ]:
# CSV로 저장 (이후 수업에서 재사용)
df_news.to_csv('news_data.csv', index=False, encoding='utf-8-sig')
print("✅ news_data.csv 저장 완료!")


### ✏️ 실습 과제 1

> **Gen AI에게 다음과 같이 요청해보세요:**
>
> *"다음 뉴스 기사 URL에서 본문 전체 텍스트를 추출하는 Python 함수를 만들어줘.  
> requests와 BeautifulSoup을 사용하고, 기사 본문만 깔끔하게 추출해줘."*
>
> 생성된 코드를 아래 셀에 붙여넣고 실행해보세요.  
> 수집한 `df_news`의 link 컬럼에 있는 URL 중 하나를 테스트해보세요.


In [ ]:
# ✏️ 여기에 Gen AI가 생성한 코드를 붙여넣으세요





---

## 3. 한국어 전처리 파이프라인

### 3.1 한국어 vs 영어, 무엇이 다른가?

영어와 한국어의 토큰화 방식은 근본적으로 다릅니다.


In [ ]:
# 영어: 띄어쓰기로 쉽게 토큰이 나뉨
english = "I love natural language processing"
print("영어 토큰화:", english.split())

# 한국어: 띄어쓰기만으로는 부족함!
korean = "나는 자연어처리를 좋아합니다"
print("한국어 단순 분리:", korean.split())
print()
print("❌ '나는' = '나' + '는(조사)' 인데, 하나의 토큰으로 취급됨")
print("❌ '좋아합니다' = '좋아하' + 'ㅂ니다' 인데, 분리가 안 됨")
print()
print("→ 한국어는 형태소 분석이 필수입니다!")


### 3.2 Kiwi 형태소 분석기

**Kiwi(Korean Intelligent Word Identifier)** 는 한국어 형태소 분석기입니다.

**형태소(Morpheme)** 란 의미를 가진 가장 작은 언어 단위입니다.

| 예시 | 형태소 분석 결과 |
|------|----------------|
| 나는 | 나/NP + 는/JX |
| 자연어처리를 | 자연어/NNG + 처리/NNG + 를/JKO |
| 좋아합니다 | 좋아하/VV + ㅂ니다/EF |

주요 품사 태그:
- **NNG**: 일반 명사 (사과, 컴퓨터)
- **NNP**: 고유 명사 (서울, 삼성)
- **VV**: 동사 (가다, 먹다)
- **VA**: 형용사 (좋다, 크다)
- **JK***: 조사 (은/는/이/가/를)


In [ ]:
# Kiwi로 형태소 분석 해보기
text = "삼성전자가 새로운 인공지능 반도체를 개발했습니다."

# tokenize()로 형태소 분석
result = kiwi.tokenize(text)

print(f"원문: {text}")
print(f"\n형태소 분석 결과:")
for token in result:
    print(f"  {token.form:10s} | {token.tag:5s} | {token.start}~{token.end}")


In [ ]:
# 여러 문장으로 테스트
test_sentences = [
    "오늘 날씨가 정말 좋네요",
    "주가가 급등하면서 투자자들이 환호했다",
    "BTS가 빌보드 차트 1위를 기록했습니다",
]


for sent in test_sentences:
    tokens = kiwi.tokenize(sent)
    # 형태소와 품사만 추출
    morphs = [(t.form, t.tag) for t in tokens]
    print(f"원문: {sent}")
    print(f"분석: {morphs}")
    print()


### 3.3 전처리 파이프라인 구축

NLP 전처리의 일반적인 순서:
1. **정제(Cleaning)**: 특수문자, URL, HTML 태그 제거
2. **형태소 분석**: 토큰화 + 품사 태깅
3. **품사 필터링**: 명사, 동사, 형용사 등 의미 있는 품사만 선택
4. **불용어 제거**: 분석에 불필요한 단어 제거
5. **결과 조합**: 전처리된 토큰을 하나의 문자열로 결합


In [ ]:
def clean_text(text):
    """
    텍스트 정제 함수
    - HTML 태그 제거
    - URL 제거
    - 특수문자 제거 (한글, 영문, 숫자, 공백만 남김)
    - 연속 공백을 하나로
    """
    # HTML 태그 제거: <b>, </b> 등
    text = re.sub(r'<[^>]+>', '', text)

    # URL 제거: http://, https:// 로 시작하는 문자열
    text = re.sub(r'https?://\S+', '', text)

    # 이메일 제거
    text = re.sub(r'\S+@\S+', '', text)

    # 특수문자 제거 (한글, 영문, 숫자, 공백만 남김)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', text)

    # 연속 공백을 하나로
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# 테스트
test = "삼성전자 <b>주가</b> 급등!! https://news.com 📈 #주식 @투자자"
print(f"원문: {test}")
print(f"정제: {clean_text(test)}")


In [ ]:
# 한국어 불용어 리스트
# NLP에서 불용어(stopwords)란 분석에 의미가 없는 단어들입니다
# 예: 조사(은, 는, 이, 가), 어미, 접속사 등

korean_stopwords = {
    # 일반적인 불용어
    '것', '등', '및', '약', '또', '를', '수', '이', '그', '저',
    '더', '때', '중', '위', '뿐', '즉', '한', '할', '감', '곳',
    # 뉴스에서 자주 나오지만 의미 없는 단어
    '기자', '뉴스', '보도', '지난', '올해', '관련', '대한', '통해',
    '이번', '현재', '최근', '오늘', '내년', '에서', '으로', '하는',
    '있는', '하고', '에서', '라고', '밝혔다', '전했다', '말했다',
}

print(f"불용어 수: {len(korean_stopwords)}개")
print(f"예시: {list(korean_stopwords)[:10]}")


In [ ]:
def preprocess_korean(text, stopwords=korean_stopwords):
    """
    한국어 텍스트 전처리 파이프라인

    1. 텍스트 정제 (특수문자, URL 등 제거)
    2. Kiwi 형태소 분석
    3. 명사, 동사, 형용사만 필터링
    4. 불용어 제거
    5. 1글자 단어 제거

    Parameters:
        text (str): 원본 텍스트
        stopwords (set): 불용어 집합

    Returns:
        str: 전처리된 토큰들을 공백으로 연결한 문자열
    """
    # 1. 텍스트 정제
    text = clean_text(text)

    if not text:
        return ""

    # 2. 형태소 분석
    tokens = kiwi.tokenize(text)

    # 3. 의미 있는 품사만 필터링
    # NNG(일반명사), NNP(고유명사), VV(동사), VA(형용사), SL(외국어)
    valid_tags = {'NNG', 'NNP', 'VV', 'VA', 'SL'}
    filtered = [t.form for t in tokens if t.tag in valid_tags]

    # 4. 불용어 제거
    filtered = [w for w in filtered if w not in stopwords]

    # 5. 1글자 단어 제거 (보통 의미가 약함)
    filtered = [w for w in filtered if len(w) > 1]

    return ' '.join(filtered)


# 테스트
test_text = "삼성전자가 새로운 인공지능 반도체를 개발했다고 기자가 보도했습니다."
print(f"원문:   {test_text}")
print(f"전처리: {preprocess_korean(test_text)}")


### 3.4 수집한 뉴스 데이터에 전처리 적용


In [ ]:
# 뉴스 데이터 로딩 (이전에 저장한 파일)
df_news = pd.read_csv('news_data.csv')
print(f"데이터 크기: {len(df_news)}개 기사")

# 제목 + 요약을 합쳐서 하나의 텍스트로 만듦
df_news['text'] = df_news['title'].fillna('') + ' ' + df_news['description'].fillna('')

# 전처리 전 샘플 확인
print("\n[전처리 전 샘플]")
print(df_news['text'].iloc[0][:200])


In [ ]:
# 전체 데이터에 전처리 적용
# 💡 apply() 함수로 모든 행에 동일한 함수를 적용할 수 있습니다
print("전처리 시작...")
df_news['processed'] = df_news['text'].apply(preprocess_korean)
print("✅ 전처리 완료!")

# 전처리 전후 비교
print("\n[전처리 전]")
print(df_news['text'].iloc[0][:200])
print("\n[전처리 후]")
print(df_news['processed'].iloc[0][:200])


In [ ]:
# 빈 문자열 제거 (전처리 후 내용이 없어진 행)
before = len(df_news)
df_news = df_news[df_news['processed'].str.len() > 0].reset_index(drop=True)
print(f"빈 행 제거: {before} → {len(df_news)}개")


### ✏️ 실습 과제 2

> **Gen AI에게 다음과 같이 요청해보세요:**
>
> *"한국어 불용어 리스트를 더 풍부하게 만들어줘.  
> 뉴스 기사 분석에 불필요한 단어 50개를 추가로 만들어줘."*
>
> 생성된 불용어를 `korean_stopwords`에 추가하고, 전처리를 다시 돌려서 결과가 달라지는지 확인해보세요.


In [ ]:
# ✏️ 여기에 확장된 불용어를 추가하고 전처리를 다시 실행해보세요





---

## 4. TF-IDF와 키워드 추출

### 4.1 TF-IDF란?

**TF-IDF (Term Frequency - Inverse Document Frequency)** 는 문서에서 단어의 중요도를 측정하는 방법입니다.

- **TF (단어 빈도)**: 특정 문서에서 해당 단어가 얼마나 자주 나오는가
- **IDF (역문서 빈도)**: 전체 문서 중 해당 단어가 나오는 문서가 얼마나 적은가

> 💡 **핵심 직관**: 모든 문서에 나오는 단어(예: "뉴스", "오늘")는 중요도가 낮고,  
> 특정 문서에만 나오는 단어(예: "반도체", "빌보드")는 중요도가 높다.

TF-IDF = TF × IDF → **"이 단어는 이 문서에서 특히 중요한가?"**


In [ ]:
# scikit-learn의 TfidfVectorizer로 TF-IDF 계산
# 💡 이미 전처리된 텍스트를 사용합니다

vectorizer = TfidfVectorizer(
    max_features=1000,    # 상위 1000개 단어만 사용
    min_df=2,             # 최소 2개 문서에 등장하는 단어만 사용
    max_df=0.8,           # 80% 이상의 문서에 등장하는 단어는 제외
)

# 전처리된 텍스트로 TF-IDF 행렬 생성
tfidf_matrix = vectorizer.fit_transform(df_news['processed'])

# 결과 확인
print(f"TF-IDF 행렬 크기: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]}개 문서, {tfidf_matrix.shape[1]}개 단어")
print(f"\n사용된 단어 예시 (처음 20개):")
feature_names = vectorizer.get_feature_names_out()
print(list(feature_names[:20]))


### 4.2 주제별 핵심 키워드 추출


In [ ]:
def get_top_keywords(query, df, tfidf_matrix, vectorizer, top_n=15):
    """
    특정 주제(query)에 해당하는 기사들의 TF-IDF 키워드를 추출하는 함수

    Parameters:
        query (str): 주제 (검색어)
        df (pd.DataFrame): 뉴스 데이터프레임
        tfidf_matrix: TF-IDF 행렬
        vectorizer: TfidfVectorizer 객체
        top_n (int): 상위 몇 개의 키워드를 추출할지

    Returns:
        dict: {단어: TF-IDF 점수} 딕셔너리
    """
    # 해당 주제의 기사 인덱스
    mask = df['query'] == query
    indices = df[mask].index.tolist()

    if not indices:
        return {}

    # 해당 기사들의 TF-IDF 벡터 평균
    topic_tfidf = tfidf_matrix[indices].mean(axis=0).A1  # 희소행렬 → 배열

    # 상위 키워드 추출
    feature_names = vectorizer.get_feature_names_out()
    top_indices = topic_tfidf.argsort()[-top_n:][::-1]

    keywords = {feature_names[i]: round(topic_tfidf[i], 4) for i in top_indices}
    return keywords


# 모든 주제별 키워드 추출
for query in df_news['query'].unique():
    keywords = get_top_keywords(query, df_news, tfidf_matrix, vectorizer)
    print(f"\n📌 [{query}] 핵심 키워드:")
    for word, score in keywords.items():
        bar = '█' * int(score * 100)
        print(f"  {word:10s} {score:.4f} {bar}")


### 4.3 워드클라우드 시각화


In [ ]:
# 한글 워드클라우드용 폰트 경로 (Colab)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

def make_wordcloud(keywords, title="워드클라우드"):
    """키워드 딕셔너리로 워드클라우드를 생성합니다"""
    wc = WordCloud(
        font_path=font_path,
        width=800,
        height=400,
        background_color='white',
        colormap='viridis',
        max_words=50,
    )
    wc.generate_from_frequencies(keywords)

    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(title, fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


# 주제별 워드클라우드 생성
for query in df_news['query'].unique():
    keywords = get_top_keywords(query, df_news, tfidf_matrix, vectorizer, top_n=30)
    make_wordcloud(keywords, title=f"'{query}' 핵심 키워드")


### 4.4 전체 데이터 단어 빈도 분석


In [ ]:
# 전체 데이터의 단어 빈도 분석
all_words = ' '.join(df_news['processed']).split()
word_counts = Counter(all_words)

# 상위 20개 단어 막대그래프
top20 = word_counts.most_common(20)
words, counts = zip(*top20)

plt.figure(figsize=(12, 6))
plt.barh(range(len(words)), counts, color='steelblue')
plt.yticks(range(len(words)), words, fontsize=12)
plt.xlabel('빈도', fontsize=12)
plt.title('전체 뉴스 데이터 - 상위 20 단어', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()  # 가장 빈도 높은 단어를 위에
plt.tight_layout()
plt.show()

print(f"\n전체 고유 단어 수: {len(word_counts):,}개")
print(f"전체 단어 수: {len(all_words):,}개")


### ✏️ 실습 과제 3

> **Gen AI에게 다음과 같이 요청해보세요:**
>
> *"Streamlit으로 뉴스 키워드 분석 앱을 만들어줘.  
> 기능: 1) 드롭다운으로 주제 선택, 2) 해당 주제의 키워드 워드클라우드 표시,  
> 3) 키워드 TOP 10 테이블 표시.  
> 데이터는 news_data.csv에서 읽고, kiwipiepy로 전처리해줘."*
>
> 생성된 코드를 `app.py`로 저장하고 실행해보세요.


In [ ]:
# ✏️ Streamlit 앱 코드를 작성하고 파일로 저장합니다
# 아래 %%writefile 매직 명령어로 파일을 만들 수 있습니다

# %%writefile app_day1.py
# (Gen AI가 생성한 Streamlit 앱 코드를 여기에 붙여넣으세요)




In [ ]:
# Streamlit 앱 실행 (Colab에서는 localtunnel 사용)
# !streamlit run app_day1.py &>/dev/null &
# !npx localtunnel --port 8501


---

## 📝 Day 1 정리

오늘 배운 것:
1. **웹 스크래핑**: requests + BeautifulSoup으로 뉴스 데이터를 직접 수집했습니다
2. **한국어 전처리**: Kiwi 형태소 분석기로 토큰화, 품사 필터링, 불용어 제거를 했습니다
3. **TF-IDF**: 텍스트를 숫자 벡터로 변환하고, 주제별 핵심 키워드를 추출했습니다
4. **워드클라우드**: 추출한 키워드를 시각화했습니다

> 💡 **내일은** 이 전처리 파이프라인을 활용해서 **감성 분석(긍정/부정 분류)** 을 해봅니다!

---
